# Food Long-Tail OOD Challenge — Vanilla ResNet Baseline

Plainest possible baseline:
1. ResNet-50 ImageNet pretrained, fine-tune on the full training set (no long-tail tricks).
2. Predict the argmax over 101 known classes for every test image.
3. Write `submission.csv`.

Expected behavior: the model can never predict the `unknown` class (101), so the ~25% OOD test samples will all be wrong — capping the achievable Top-1 accuracy at roughly 75%. This baseline establishes the floor; better submissions add long-tail and OOD handling on top.

## 1. Setup

In [4]:
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

# 直接拉取你们的比赛数据
!kaggle competitions download -c ow-food

# 将下载好的压缩包解压到专门的文件夹中
!unzip -q ow-food.zip -d /content/Food-Classification

print("✅ 数据集极速下载与解压完毕！")

100% 853M/853M [00:52<00:00, 16.9MB/s]

✅ 数据集极速下载与解压完毕！


In [5]:
import time
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from PIL import Image
from torch.utils.data import DataLoader, Dataset
from torchvision import models, transforms
from tqdm.auto import tqdm

# 数据根目录与提交输出路径
DATA_ROOT = Path('/content/Food-Classification')
OUT_SUB = DATA_ROOT / 'submission(6).csv'

# 已知类数：模型只学 0..100，OOD（101）由后处理判定
NUM_KNOWN = 101
IMG_SIZE = 224          # ResNet 默认输入；图像本身是 256，会被 resize 到 224
BATCH_SIZE = 64
NUM_WORKERS = 4
EPOCHS = 5              # baseline 跑通即可，不追求收敛
LR = 1e-3
SEED = 42

# 自动选设备：优先 CUDA → Apple MPS → CPU
if torch.cuda.is_available():
    DEVICE = torch.device('cuda')
elif torch.backends.mps.is_available():
    DEVICE = torch.device('mps')
else:
    DEVICE = torch.device('cpu')
print('Device:', DEVICE)

torch.manual_seed(SEED); np.random.seed(SEED)

Device: cuda


## 2. Dataset

In [6]:
IMNET_MEAN = [0.485, 0.456, 0.406]
IMNET_STD = [0.229, 0.224, 0.225]

train_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(IMNET_MEAN, IMNET_STD),
])
eval_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(IMNET_MEAN, IMNET_STD),
])

class FoodDataset(Dataset):
    """从 csv (image_id[, label]) + 图像目录加载样本。
    has_label=False 用于测试集，__getitem__ 第二项返回 image_id 以便回填。"""

    def __init__(self, df, img_dir, transform, has_label):
        self.df = df.reset_index(drop=True)
        self.img_dir = Path(img_dir)
        self.transform = transform
        self.has_label = has_label

    def __len__(self):
        return len(self.df)

    def __getitem__(self, i):
        row = self.df.iloc[i]
        img = Image.open(self.img_dir / f"{row['image_id']}.jpg").convert('RGB')
        img = self.transform(img)
        if self.has_label:
            return img, int(row['label'])
        return img, row['image_id']

# 现在程序知道 DATA_ROOT 是什么了，再安全地读取 CSV
train_df = pd.read_csv(DATA_ROOT / 'train.csv')
test_df = pd.read_csv(DATA_ROOT / 'test.csv')
print(f'✅ 表格读取成功！train: {len(train_df)}, test: {len(test_df)}')

# 加载图片数据集
tr_ds = FoodDataset(train_df, DATA_ROOT / 'train_images' / 'train_images', train_tf, has_label=True)
test_ds = FoodDataset(test_df, DATA_ROOT / 'test_images' / 'test_images', eval_tf, has_label=False)

# pin_memory 仅在 CUDA 下能加速，MPS/CPU 无效
tr_loader = DataLoader(tr_ds, batch_size=BATCH_SIZE, shuffle=True,
                       num_workers=NUM_WORKERS, pin_memory=(DEVICE.type=='cuda'))
# 推理时 batch 翻倍，无需反向传播显存压力小
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE*2, shuffle=False,
                         num_workers=NUM_WORKERS, pin_memory=(DEVICE.type=='cuda'))
print('✅ 数据流水线组装完毕，可以开始训练了！')

✅ 表格读取成功！train: 32856, test: 20179
✅ 数据流水线组装完毕，可以开始训练了！


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()


## 3. Model

In [7]:
import torch.nn as nn
import torchvision.models as models

# === ConvNeXt Tiny 模型 ===
model = models.convnext_tiny(pretrained=True)

# 修改分类头输出 NUM_KNOWN 类
in_features = model.classifier[2].in_features
model.classifier[2] = nn.Linear(in_features, NUM_KNOWN)

model = model.to(DEVICE)
print("Using model: ConvNeXt‑Tiny")

/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ConvNeXt_Tiny_Weights.IMAGENET1K_V1`. You can also use `weights=ConvNeXt_Tiny_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Downloading: "https://download.pytorch.org/models/convnext_tiny-983f1562.pth" to /root/.cache/torch/hub/checkpoints/convnext_tiny-983f1562.pth


100%|██████████| 109M/109M [00:00<00:00, 144MB/s] 


Using model: ConvNeXt‑Tiny


## 4. Train

In [8]:
# =====================================================================
# 🎯 核心大招：计算并注入长尾逆频率权重（带平方根平滑）
# =====================================================================

print("⚖️ 正在计算长尾类别分布权重...")

class_counts = train_df['label'].value_counts().sort_index().values

# √ 平滑：比原始 1/N 更稳
weights = 1.0 / (np.sqrt(class_counts) + 1.0)
weights = weights / np.sum(weights) * NUM_KNOWN
class_weights = torch.FloatTensor(weights).to(DEVICE)

criterion = nn.CrossEntropyLoss(weight=class_weights)
print("✅ 平衡交叉熵损失 (Balanced Cross Entropy) 加载完毕！")

# √ 微调学习率更小更稳定
optimizer = torch.optim.AdamW(model.parameters(), lr=5e-5, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

for epoch in range(1, EPOCHS + 1):
    model.train()
    t0 = time.time()
    total, correct, loss_sum = 0, 0, 0.0
    bar = tqdm(tr_loader, desc=f'epoch {epoch}/{EPOCHS}', leave=False)

    for x, y in bar:
        x, y = x.to(DEVICE, non_blocking=True), y.to(DEVICE, non_blocking=True)
        logits = model(x)
        loss = criterion(logits, y)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        loss_sum += loss.item() * x.size(0)
        correct += (logits.argmax(1) == y).sum().item()
        total += x.size(0)
        bar.set_postfix(loss=loss_sum/total, acc=correct/total)

    scheduler.step()
    print(f"Epoch {epoch}/{EPOCHS} | loss {loss_sum/total:.4f} | train acc {correct/total:.4f} | {time.time()-t0:.1f}s")

⚖️ 正在计算长尾类别分布权重...
✅ 平衡交叉熵损失 (Balanced Cross Entropy) 加载完毕！


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:432: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()


epoch 1/5:   0%|          | 0/514 [00:00<?, ?it/s]

Epoch 1/5 | loss 2.8657 | train acc 0.4959 | 573.9s


epoch 2/5:   0%|          | 0/514 [00:00<?, ?it/s]

...

Epoch 3/5 | loss 1.0843 | train acc 0.7935 | 576.5s


epoch 4/5:   0%|          | 0/514 [00:00<?, ?it/s]

Epoch 4/5 | loss 0.8252 | train acc 0.8434 | 575.3s


epoch 5/5:   0%|          | 0/514 [00:00<?, ?it/s]

Epoch 5/5 | loss 0.6993 | train acc 0.8681 | 576.9s


In [9]:
import torch
import torch.nn.functional as F
import numpy as np
from tqdm.auto import tqdm

print("🔍 正在提取训练集特征，用于计算每类中心...")

model.eval()
features_per_class = [[] for _ in range(NUM_KNOWN)]

with torch.no_grad():
    for x, y in tqdm(tr_loader, desc="Extract train features"):
        x = x.to(DEVICE)
        y = y.to(DEVICE)

        # 提取分类 head 前的特征
        feats = model.features(x)             # [B, C, H, W]
        feats = F.adaptive_avg_pool2d(feats, 1)
        feats = torch.flatten(feats, 1)       # [B, 768]
        feats = F.normalize(feats, dim=1)     # 归一化更稳定

        for i in range(x.size(0)):
            label = y[i].item()
            features_per_class[label].append(feats[i].cpu().numpy())

# 计算每类中心
class_centers = []
for c in range(NUM_KNOWN):
    if len(features_per_class[c]) == 0:
        center = np.zeros_like(features_per_class[0][0])
    else:
        center = np.stack(features_per_class[c]).mean(axis=0)
    class_centers.append(center)

class_centers = np.stack(class_centers)     # [101, feat_dim]
class_centers = torch.tensor(class_centers).to(DEVICE)
class_centers = F.normalize(class_centers, dim=1)

print("✅ 特征中心计算完成！")

🔍 正在提取训练集特征，用于计算每类中心...


Extract train features:   0%|          | 0/514 [00:00<?, ?it/s]

Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7beb430d65c0>
Traceback (most recent call last):
Exception ignored in:   File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1709, in __del__
Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7beb430d65c0>Exception ignored in: 
    <function _MultiProcessingDataLoaderIter.__del__ at 0x7beb430d65c0>Traceback (most recent call last):

  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1709, in __del__
<function _MultiProcessingDataLoaderIter.__del__ at 0x7beb430d65c0>Traceback (most recent call last):
self._shutdown_workers()
      File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1709, in __del__
Traceback (most recent call last):
self._shutdown_workers()
    
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1709, in __del__
  File "/usr/local/lib/pyt

✅ 特征中心计算完成！


## 5. Predict on test set

In [10]:
import pandas as pd

print("🚀 Running test prediction with feature distance...")

model.eval()
all_ids = []
all_preds = []

with torch.no_grad():
    for x, ids in tqdm(test_loader, desc="Predict test features"):
        x = x.to(DEVICE)

        feats = model.features(x)
        feats = F.adaptive_avg_pool2d(feats, 1)
        feats = torch.flatten(feats, 1)
        feats = F.normalize(feats, dim=1)

        # cosine similarity
        sims = feats @ class_centers.T        # [batch, 101]
        max_sim, pred = sims.max(dim=1)

        threshold = np.quantile(max_sim.cpu().numpy(), 0.25)
        pred = torch.where(max_sim <= threshold, torch.full_like(pred, 101), pred)

        all_ids.extend(ids)
        all_preds.extend(pred.cpu().numpy().tolist())

submission = pd.DataFrame({"image_id": all_ids, "label": all_preds})
order = pd.read_csv(DATA_ROOT / "test.csv")["image_id"].tolist()
submission = submission.set_index("image_id").loc[order].reset_index()

OUT_SUB = DATA_ROOT / "submission_convnext_tiny.csv"
submission.to_csv(OUT_SUB, index=False)
print(f"📦 Saved: {OUT_SUB}")

🚀 Running test prediction with feature distance...


Predict test features:   0%|          | 0/158 [00:00<?, ?it/s]

📦 Saved: /content/Food-Classification/submission_convnext_tiny.csv


## 6. Submission